<a href="https://colab.research.google.com/github/anuragN2107/Radianttrust-Diagnostic-System/blob/main/Automated_Chest_X_Ray_Diagnostic_%26_Explainability_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Name:
 RadiantTrust AI (Automated Chest X-Ray Diagnostic & Explainability System)

# Business Problem:
 Deep learning models can achieve high accuracy in medical imaging, but clinicians reject them because they operate as a "black box." A doctor cannot trust a raw percentage score when a patient's life is on the line. If a model predicts "Pneumonia" without proving why, it is useless in a clinical environment.

# Objectives:
* Build a robust Convolutional Neural Network (CNN) using Transfer Learning to classify chest X-rays into Normal vs. Pneumonia.
* Implement Grad-CAM (Gradient-weighted Class Activation Mapping) to overlay a visual heatmap on the pathology, showing exactly where the network is focusing.
* Package the pipeline into an intuitive web interface for immediate image uploads and spatial verification.  

# Architecture Strategy
Instead of reinventing the wheel with a basic CNN that struggles to generalize, we utilize ResNet18 via Transfer Learning. ResNet’s residual connections (skip connections) prevent vanishing gradients, allowing the network to extract rich, low-level edge features and high-level structural patterns crucial for medical scans.



# The Explainability Engine (Grad-CAM)
Standard neural networks pool spatial information into a flat vector before classification, losing the "where" information. Grad-CAM fixes this. By taking the gradients of the target score with respect to the final convolutional layer's feature maps, we calculate an importance weight for each feature map. This produces a localization map highlighting the precise regions that drove the model's decision.

#Tools & Requirements
* Compute: Google Colab (Free T4 GPU enabled)
* Core Libraries: PyTorch, torchvision (Model building & optimization)
* Data Processing: OpenCV, PIL, NumPy, Pandas
* Deployment: Streamlit (For building the rapid web UI)

#Step 1: Environment Setup & Mock Data Generation

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image, ImageDraw
import cv2
import matplotlib.pyplot as plt

In [2]:
#Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [3]:
#Create local directory mock structure
os.makedirs("dataset/train/normal", exist_ok=True)
os.makedirs("dataset/train/pneumonia", exist_ok=True)
os.makedirs("dataset/val/normal", exist_ok=True)
os.makedirs("dataset/val/pneumonia", exist_ok=True)

In [4]:
def generate_mock_xray(path, label):
    """Generates a synthetic image resembling a chest X-ray with lung opacity."""
    # Create dark grey chest cavity base
    img_arr = np.ones((224, 224), dtype=np.uint8) * 30
    img = Image.fromarray(img_arr).convert("RGB")
    draw = ImageDraw.Draw(img)

    # Draw two translucent-looking lung fields
    draw.ellipse([40, 40, 100, 200], fill=(80, 80, 80))
    draw.ellipse([124, 40, 184, 200], fill=(80, 80, 80))

    # If pneumonia, inject a localized 'cloudy' patch
    if label == "pneumonia":
        draw.ellipse([50, 100, 90, 150], fill=(180, 180, 180))

    img.save(path)

# Generate 40 training & 10 validation samples
for i in range(20):
    generate_mock_xray(f"dataset/train/normal/norm_{i}.png", "normal")
    generate_mock_xray(f"dataset/train/pneumonia/pneu_{i}.png", "pneumonia")
for i in range(5):
    generate_mock_xray(f"dataset/val/normal/norm_{i}.png", "normal")
    generate_mock_xray(f"dataset/val/pneumonia/pneu_{i}.png", "pneumonia")

print(" Mock dataset constructed successfully!")

 Mock dataset constructed successfully!


#Step 2: Data Pipelines & Transformations
We set up transformations to convert standard imagery to tensor format, normalization matrices matching ImageNet baselines, and PyTorch DataLoaders.

In [5]:
# Image transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

from torchvision.datasets import ImageFolder

train_dataset = ImageFolder("dataset/train", transform=data_transforms['train'])
val_dataset = ImageFolder("dataset/val", transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Class Mapping: {train_dataset.class_to_idx}")

Class Mapping: {'normal': 0, 'pneumonia': 1}


#Step 3: Transfer Learning Model Setup
We modify the ResNet18 architecture by freezing its initial convolutional layers, which serve as feature extractors. Additionally, we replace the final fully connected classification layer with a new layer designed for binary classification, enabling the model to address our specific two-class challenge effectively.

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze lower layers to preserve ImageNet weights
for param in model.parameters():
    param.requires_grad = False

# Swap out the final linear layer for binary classification
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 223MB/s]


# Step 4: Model Training Loop
We implement a standard training loop that iterates over our generated dataset. During each epoch, only the weights of the final fully connected layer are updated, allowing the model to learn from the data while keeping the early feature extractors fixed.

In [7]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss, running_corrects = 0.0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)
    print(f"Epoch {epoch+1}/{epochs} -> Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}")

# Save the trained weights
torch.save(model.state_dict(), "pneumonia_resnet18.pth")
print("Model saved as pneumonia_resnet18.pth")

Epoch 1/5 -> Loss: 0.7399 | Accuracy: 0.4500
Epoch 2/5 -> Loss: 0.4228 | Accuracy: 0.8500
Epoch 3/5 -> Loss: 0.2608 | Accuracy: 1.0000
Epoch 4/5 -> Loss: 0.2673 | Accuracy: 0.9000
Epoch 5/5 -> Loss: 0.0983 | Accuracy: 1.0000
Model saved as pneumonia_resnet18.pth


#Step 5: Implementation of the Grad-CAM Engine
This block registers hooks directly into ResNet18's final convolutional block (layer4) to calculate attention matrices on demand.

In [8]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Hook activations and gradients
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate_heatmap(self, input_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0][class_idx]
        score.backward()

        # Global Average Pooling of gradients
        weights = torch.mean(self.gradients, dim=[2, 3])[0]

        # Weighted combination of forward activation maps
        heatmap = torch.zeros(self.activations.shape[2:], dtype=torch.float32)
        for i, w in enumerate(weights):
            heatmap += w * self.activations[0, i, :, :]

        # ReLU activation
        heatmap = np.maximum(heatmap.cpu().numpy(), 0)
        if np.max(heatmap) != 0:
            heatmap /= np.max(heatmap) # Normalize

        return heatmap

print("Grad-CAM Engine compiled successfully.")

Grad-CAM Engine compiled successfully.
